# Open RAG Eval - Reference-free Evaluation

Evaluate without golden answers: grade retrieved passages (UMBRELA-style) and check answer nuggets.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

**Extra requirements:** Optional: `pip install open-rag-eval` for the full Vectara framework

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

[open-rag-eval](https://github.com/vectara/open-rag-eval) evaluates RAG **without ground-truth answers** by combining: retrieval relevance (UMBRELA), nugget coverage (AutoNuggetizer), citation checks and hallucination detection. This notebook re-implements the core ideas locally so you can see how they work.

## 1. UMBRELA-style retrieval grading
Each retrieved passage gets 0-3: 0 unrelated, 1 related but no answer, 2 partial answer, 3 dedicated answer.

In [ ]:
def umbrela(query, passage):
    out = llm(
        "Grade the passage for the query: 0 = unrelated, 1 = related but does not answer, "
        "2 = partially answers, 3 = dedicated, exact answer. Reply with the digit only.\n\n"
        f"Query: {query}\n\nPassage: {passage}"
    )
    m = re.search(r"[0-3]", out)
    return int(m.group()) if m else 0

query = "How does hybrid search merge keyword and vector results?"
hits = store.search(query, 5)
grades = [umbrela(query, h["text"]) for h in hits]
print("grades:", grades, "-> mean", sum(grades) / len(grades))

## 2. Nugget evaluation
Break the *information need* into atomic nuggets (facts a good answer must contain), then check which ones the answer covers.

In [ ]:
nuggets = llm_json(f"List 4 atomic facts a complete answer to this question must contain. JSON list of strings.\n\nQuestion: {query}") or []
response = answer(query, [h["text"] for h in hits])

covered = {n: llm_yes(f"Answer: {response}\n\nDoes the answer clearly state this fact: {n}") for n in nuggets}
for n, ok in covered.items(): print("[x]" if ok else "[ ]", n)
print(f"nugget recall: {sum(covered.values()) / max(len(covered), 1):.2f}")

## 3. Hallucination / citation check
Split the answer into sentences and verify each one against the retrieved context.

In [ ]:
ctx = "\n".join(h["text"] for h in hits)
sentences = split_sentences(response)
supported = [llm_yes(f"Context:\n{ctx}\n\nClaim: {s}\nIs the claim supported by the context?") for s in sentences]
print(f"{sum(supported)}/{len(supported)} sentences supported by retrieved context")